In [1]:
# Cell 1 — Install Dependencies

print("Installing required packages...")

!pip install smplx trimesh scipy tqdm

print("Installation complete.")

Installing required packages...

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Installation complete.


In [2]:
# Cell 2 — Imports and Setup

print("Loading imports and setting up environment...")

import os
import numpy as np
import torch
import trimesh
import smplx
from tqdm import tqdm

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")

Loading imports and setting up environment...
Using device: cuda


In [4]:
# Cell 3 — Set Paths

print("Setting file paths...")

# SMPL path (your correct path)
smpl_model_path = os.path.join(
    "/workspace",
    "body measurements and weight cal",
    "lankesha",
    "models",
    "smpl",
    "SMPL_NEUTRAL.pkl"
)

# Your mesh path (UPDATE THIS)
target_mesh_path = "first.glb"

# Output directory
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)

print(f"SMPL path: {smpl_model_path}")
print(f"Mesh path: {target_mesh_path}")
print(f"Output dir: {output_dir}")

# Check existence
print("SMPL exists:", os.path.exists(smpl_model_path))
print("Mesh exists:", os.path.exists(target_mesh_path))

Setting file paths...
SMPL path: /workspace/body measurements and weight cal/lankesha/models/smpl/SMPL_NEUTRAL.pkl
Mesh path: first.glb
Output dir: ./output
SMPL exists: True
Mesh exists: True


In [5]:
# Cell 4 — Load Target Mesh

print("Loading target mesh...")

target_mesh = trimesh.load(target_mesh_path, process=False)

# Handle scene
if isinstance(target_mesh, trimesh.Scene):
    print("Converting scene to mesh...")
    target_mesh = trimesh.util.concatenate(
        [g for g in target_mesh.geometry.values()]
    )

target_mesh.remove_unreferenced_vertices()

print(f"Vertices: {len(target_mesh.vertices)}")
print(f"Faces: {len(target_mesh.faces)}")

Loading target mesh...
Converting scene to mesh...
Vertices: 18439
Faces: 36874


In [9]:
# Cell 5 — Patch Python 3.11 and NumPy compatibility for chumpy

# Print start message
print("Applying compatibility patches for chumpy...")

# Import inspect for monkey patching
import inspect

# Import numpy for alias patching
import numpy as np

# Import namedtuple to recreate ArgSpec
from collections import namedtuple

# -----------------------------
# Patch inspect.getargspec
# -----------------------------

# Check whether inspect.getargspec is missing
if not hasattr(inspect, "getargspec"):
    # Print patch status
    print("Patching inspect.getargspec...")

    # Create compatible ArgSpec structure
    ArgSpec = namedtuple("ArgSpec", ["args", "varargs", "keywords", "defaults"])

    # Define replacement function
    def getargspec(func):
        # Get full argument spec
        spec = inspect.getfullargspec(func)
        
        # Return old-style ArgSpec
        return ArgSpec(
            args=spec.args,
            varargs=spec.varargs,
            keywords=spec.varkw,
            defaults=spec.defaults
        )

    # Apply patch
    inspect.getargspec = getargspec

    # Print success
    print("inspect.getargspec patch applied.")
else:
    # Print status
    print("inspect.getargspec already exists.")

# -----------------------------
# Patch removed NumPy aliases
# -----------------------------

# Create missing NumPy aliases if needed
numpy_aliases = {
    "bool": bool,
    "int": int,
    "float": float,
    "complex": complex,
    "object": object,
    "str": str
}

# Add unicode alias safely
try:
    unicode
except NameError:
    unicode = str

# Add unicode into alias dictionary
numpy_aliases["unicode"] = unicode

# Loop through aliases
for alias_name, alias_value in numpy_aliases.items():
    # Add alias only if missing
    if not hasattr(np, alias_name):
        setattr(np, alias_name, alias_value)
        print(f"Patched np.{alias_name}")
    else:
        print(f"np.{alias_name} already exists")

# Print completion
print("All compatibility patches applied successfully.")

Applying compatibility patches for chumpy...
inspect.getargspec already exists.
np.bool already exists
Patched np.int
Patched np.float
Patched np.complex
Patched np.object
Patched np.str
Patched np.unicode
All compatibility patches applied successfully.


/tmp/ipykernel_5325/1259161106.py:75: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, alias_name):
/tmp/ipykernel_5325/1259161106.py:75: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, alias_name):


In [10]:
# Cell 6 — Load SMPL Model

# Print start message
print("Loading SMPL model...")

# Import os for path checks
import os

# Import smplx
import smplx

# Check whether the SMPL file exists
if not os.path.exists(smpl_model_path):
    raise FileNotFoundError(f"SMPL file not found: {smpl_model_path}")

# Print the file being used
print(f"Using SMPL file: {smpl_model_path}")

# Load the SMPL model
smpl_model = smplx.create(
    smpl_model_path,
    model_type="smpl",
    gender="neutral",
    batch_size=1
).to(device)

# Get SMPL faces
smpl_faces = smpl_model.faces

# Print success info
print("SMPL model loaded successfully.")
print(f"Number of SMPL faces: {len(smpl_faces)}")

Loading SMPL model...
Using SMPL file: /workspace/body measurements and weight cal/lankesha/models/smpl/SMPL_NEUTRAL.pkl
SMPL model loaded successfully.
Number of SMPL faces: 13776


In [11]:
# Cell 7 — Prepare Target Points and Optimization Variables

# Print start message
print("Preparing target points and optimization variables...")

# Set the number of target surface points
num_target_points = 3000

# Sample points from the target mesh surface
target_points_np, _ = trimesh.sample.sample_surface(target_mesh, num_target_points)

# Convert target points to torch tensor
target_points = torch.tensor(target_points_np, dtype=torch.float32, device=device)

# Print target points shape
print(f"Target points shape: {target_points.shape}")

# Compute target center
target_center = target_points.mean(dim=0, keepdim=True)

# Print target center
print(f"Target center: {target_center.detach().cpu().numpy()}")

# Initialize shape parameters
betas = torch.zeros((1, 10), dtype=torch.float32, device=device, requires_grad=True)

# Initialize body pose parameters
body_pose = torch.zeros((1, 69), dtype=torch.float32, device=device, requires_grad=True)

# Initialize global orientation
global_orient = torch.zeros((1, 3), dtype=torch.float32, device=device, requires_grad=True)

# Initialize translation using target center
transl = target_center.clone().detach().requires_grad_(True)

# Initialize global scale
scale = torch.tensor([1.0], dtype=torch.float32, device=device, requires_grad=True)

# Print variable status
print("Optimization variables initialized.")
print(f"betas shape: {betas.shape}")
print(f"body_pose shape: {body_pose.shape}")
print(f"global_orient shape: {global_orient.shape}")
print(f"transl shape: {transl.shape}")
print(f"Initial scale: {scale.item():.4f}")

Preparing target points and optimization variables...
Target points shape: torch.Size([3000, 3])
Target center: [[ 0.02209605 -0.90201074 -0.01160805]]
Optimization variables initialized.
betas shape: torch.Size([1, 10])
body_pose shape: torch.Size([1, 69])
global_orient shape: torch.Size([1, 3])
transl shape: torch.Size([1, 3])
Initial scale: 1.0000


In [12]:
# Cell 8 — Define Helper Functions

# Print start message
print("Defining helper functions...")

# Define a helper function to normalize a vector
def normalize_vector(v):
    # Compute vector norm
    norm = np.linalg.norm(v)
    
    # Avoid division by zero
    if norm < 1e-8:
        print("normalize_vector: very small norm detected, returning input vector.")
        return v
    
    # Return normalized vector
    return v / norm


# Define a differentiable chamfer-like loss using torch.cdist
def bidirectional_chamfer_loss(points_a, points_b):
    # Compute pairwise distance matrix
    dist_matrix = torch.cdist(points_a, points_b)
    
    # Compute minimum distances from A to B
    min_a_to_b = dist_matrix.min(dim=1)[0]
    
    # Compute minimum distances from B to A
    min_b_to_a = dist_matrix.min(dim=0)[0]
    
    # Compute final loss
    loss = min_a_to_b.mean() + min_b_to_a.mean()
    
    # Return loss
    return loss


# Define a helper to get SMPL vertices and joints
def get_smpl_output():
    # Check that the model exists
    if "smpl_model" not in globals():
        raise RuntimeError("smpl_model is not defined. Run the SMPL loading cell first.")
    
    # Check required optimization variables
    required_vars = ["betas", "body_pose", "global_orient", "transl", "scale"]
    
    # Loop through required variables
    for var_name in required_vars:
        # Raise an error if missing
        if var_name not in globals():
            raise RuntimeError(f"{var_name} is not defined. Run the initialization cell first.")
    
    # Run the SMPL model
    output = smpl_model(
        betas=betas,
        body_pose=body_pose,
        global_orient=global_orient,
        transl=None
    )
    
    # Extract vertices
    verts = output.vertices[0]
    
    # Extract joints
    joints = output.joints[0]
    
    # Apply scale and translation to vertices
    verts = verts * scale + transl
    
    # Apply scale and translation to joints
    joints = joints * scale + transl
    
    # Return transformed vertices and joints
    return verts, joints


# Define a helper function to compute approximate body height
def compute_height(vertices_np):
    # Compute minimum coordinates
    vmin = vertices_np.min(axis=0)
    
    # Compute maximum coordinates
    vmax = vertices_np.max(axis=0)
    
    # Compute bounding box size
    bbox_size = vmax - vmin
    
    # Return largest dimension as approximate height
    return float(bbox_size.max())


# Define a helper to compute 3D point distance
def point_distance(a, b):
    # Return Euclidean distance
    return float(np.linalg.norm(a - b))


# Define a helper to compute circumference from a sliced mesh
def section_circumference(mesh, point_on_plane, plane_normal):
    # Print start message
    print("Trying to compute circumference from mesh section...")
    
    # Define small offsets for robustness
    offsets = [0.0, 0.005, -0.005, 0.01, -0.01, 0.02, -0.02]
    
    # Try each offset
    for off in offsets:
        # Shift point slightly along plane normal
        shifted_point = point_on_plane + plane_normal * off
        
        # Slice the mesh
        section = mesh.section(
            plane_origin=shifted_point,
            plane_normal=plane_normal
        )
        
        # Continue if no section found
        if section is None:
            continue
        
        # Convert section to 2D planar representation
        planar_section, _ = section.to_planar()
        
        # Compute total section length
        length = float(planar_section.length)
        
        # Print success message
        print(f"Section found at offset {off:.4f}, circumference = {length:.4f}")
        
        # Return circumference
        return length
    
    # Print failure message
    print("Could not compute valid section circumference.")
    
    # Return None if no section found
    return None


# Print completion message
print("Helper functions defined successfully.")

Defining helper functions...
Helper functions defined successfully.


In [13]:
# Cell 9 — Check Required Variables Before Fitting

# Print start message
print("Checking required variables before SMPL fitting...")

# List required variables
required_vars = [
    "smpl_model",
    "smpl_faces",
    "target_points",
    "betas",
    "body_pose",
    "global_orient",
    "transl",
    "scale",
    "device"
]

# Loop through variables and print status
for var_name in required_vars:
    exists = var_name in globals()
    print(f"{var_name}: {exists}")

# Create list of missing variables
missing_vars = [var_name for var_name in required_vars if var_name not in globals()]

# Raise error if anything is missing
if len(missing_vars) > 0:
    raise RuntimeError(f"Missing required variables: {missing_vars}")

# Print success message
print("All required variables are available.")

Checking required variables before SMPL fitting...
smpl_model: True
smpl_faces: True
target_points: True
betas: True
body_pose: True
global_orient: True
transl: True
scale: True
device: True
All required variables are available.


In [14]:
# Cell 10 — Optimize SMPL to Fit the Target Mesh

# Print start message
print("Starting SMPL fitting...")

# Create optimizer
optimizer = torch.optim.Adam(
    [betas, body_pose, global_orient, transl, scale],
    lr=0.02
)

# Set number of iterations
num_iters = 300

# Set number of sampled SMPL vertices per iteration
num_smpl_points = 2000

# Print optimization settings
print(f"Number of iterations: {num_iters}")
print(f"Number of sampled SMPL points: {num_smpl_points}")
print(f"Target points shape: {target_points.shape}")

# Start optimization loop
for it in tqdm(range(num_iters)):
    # Reset gradients
    optimizer.zero_grad()
    
    # Get current SMPL vertices and joints
    smpl_verts, smpl_joints = get_smpl_output()
    
    # Randomly sample vertices from SMPL mesh
    rand_idx = torch.randperm(smpl_verts.shape[0], device=device)[:num_smpl_points]
    
    # Select sampled points
    smpl_points = smpl_verts[rand_idx]
    
    # Compute fit loss
    fit_loss = bidirectional_chamfer_loss(smpl_points, target_points)
    
    # Regularize shape parameters
    beta_reg = 0.001 * (betas ** 2).mean()
    
    # Regularize body pose
    pose_reg = 0.001 * (body_pose ** 2).mean()
    
    # Regularize global orientation
    orient_reg = 0.0005 * (global_orient ** 2).mean()
    
    # Regularize scale
    scale_reg = 0.001 * ((scale - 1.0) ** 2).mean()
    
    # Compute total loss
    loss = fit_loss + beta_reg + pose_reg + orient_reg + scale_reg
    
    # Backpropagate
    loss.backward()
    
    # Update parameters
    optimizer.step()
    
    # Print progress every 20 iterations
    if it % 20 == 0 or it == num_iters - 1:
        print(
            f"Iter {it:03d} | "
            f"Total Loss: {loss.item():.6f} | "
            f"Fit Loss: {fit_loss.item():.6f} | "
            f"Scale: {scale.item():.4f}"
        )

# Print completion message
print("SMPL fitting completed successfully.")

Starting SMPL fitting...
Number of iterations: 300
Number of sampled SMPL points: 2000
Target points shape: torch.Size([3000, 3])


  4%|▍         | 13/300 [00:00<00:10, 27.16it/s]

Iter 000 | Total Loss: 0.254880 | Fit Loss: 0.254880 | Scale: 0.9800
Iter 020 | Total Loss: 0.123769 | Fit Loss: 0.123610 | Scale: 0.8007


 18%|█▊        | 54/300 [00:01<00:03, 74.85it/s]

Iter 040 | Total Loss: 0.099109 | Fit Loss: 0.098855 | Scale: 0.9814


...

Iter 220 | Total Loss: 0.082651 | Fit Loss: 0.081467 | Scale: 1.1022
Iter 240 | Total Loss: 0.081116 | Fit Loss: 0.079818 | Scale: 1.1079


 93%|█████████▎| 279/300 [00:03<00:00, 102.84it/s]

Iter 260 | Total Loss: 0.080855 | Fit Loss: 0.079481 | Scale: 1.1093
Iter 280 | Total Loss: 0.079030 | Fit Loss: 0.077584 | Scale: 1.1173


100%|██████████| 300/300 [00:03<00:00, 83.07it/s] 


Iter 299 | Total Loss: 0.080038 | Fit Loss: 0.078535 | Scale: 1.1182
SMPL fitting completed successfully.


In [15]:
# Cell 11 — Get Final SMPL Output and Save Fitted Mesh

# Print start message
print("Preparing final fitted SMPL output...")

# Get final vertices and joints
final_verts_torch, final_joints_torch = get_smpl_output()

# Convert vertices to numpy
final_verts = final_verts_torch.detach().cpu().numpy()

# Convert joints to numpy
final_joints = final_joints_torch.detach().cpu().numpy()

# Print shapes
print(f"Final vertices shape: {final_verts.shape}")
print(f"Final joints shape: {final_joints.shape}")

# Create fitted SMPL mesh
fitted_smpl_mesh = trimesh.Trimesh(
    vertices=final_verts,
    faces=smpl_faces,
    process=False
)

# Create output path
fitted_mesh_path = os.path.join(output_dir, "fitted_smpl.obj")

# Save fitted mesh
fitted_smpl_mesh.export(fitted_mesh_path)

# Print save path
print(f"Fitted SMPL mesh saved to: {fitted_mesh_path}")

# Compute fitted height
fitted_height = compute_height(final_verts)

# Print fitted height
print(f"Estimated fitted body height: {fitted_height:.4f}")

Preparing final fitted SMPL output...
Final vertices shape: (6890, 3)
Final joints shape: (45, 3)
Fitted SMPL mesh saved to: ./output/fitted_smpl.obj
Estimated fitted body height: 1.6862


In [18]:
# Cell 12 — Define Joint Indices for Final Measurements

# Print start message
print("Defining joint indices for final measurements...")

# Define important SMPL joint indices
PELVIS = 0
LEFT_HIP = 1
RIGHT_HIP = 2
LEFT_KNEE = 4
RIGHT_KNEE = 5
NECK = 12
LEFT_SHOULDER = 16
RIGHT_SHOULDER = 17

# Print completion message
print("Joint indices defined successfully.")

Defining joint indices for final measurements...
Joint indices defined successfully.


In [19]:
# Cell 13 — Define Helper Functions for Circumference Measurement

# Print start message
print("Defining helper functions for circumference measurement...")

# Define a helper function to normalize a vector
def normalize_vector(v):
    # Compute vector norm
    norm = np.linalg.norm(v)
    
    # Avoid division by zero
    if norm < 1e-8:
        print("Very small norm detected. Returning original vector.")
        return v
    
    # Return normalized vector
    return v / norm


# Define a helper function to compute circumference from a mesh section
def section_circumference(mesh, point_on_plane, plane_normal):
    # Print section start message
    print("Trying to compute section circumference...")
    
    # Define small offsets for robust slicing
    offsets = [0.0, 0.003, -0.003, 0.006, -0.006, 0.01, -0.01, 0.015, -0.015]
    
    # Loop through offsets
    for off in offsets:
        # Shift the slicing point slightly along the normal
        shifted_point = point_on_plane + plane_normal * off
        
        # Slice the mesh
        section = mesh.section(
            plane_origin=shifted_point,
            plane_normal=plane_normal
        )
        
        # Continue if no section found
        if section is None:
            continue
        
        # Convert section to planar coordinates
        planar_section, _ = section.to_planar()
        
        # Get total path length
        circumference = float(planar_section.length)
        
        # Print success message
        print(f"Section found at offset {off:.4f} with circumference {circumference:.4f}")
        
        # Return the circumference
        return circumference
    
    # Print failure message
    print("Could not compute valid section circumference.")
    
    # Return None if no section found
    return None


# Define a helper function to convert measurement to centimeters
def to_cm(value_in_model_units):
    # Convert meters to centimeters
    return value_in_model_units * 100.0


# Print completion message
print("Helper functions defined successfully.")

Defining helper functions for circumference measurement...
Helper functions defined successfully.


In [20]:
# Cell 14 — Compute Final Circumference Measurements

# Print start message
print("Computing final circumference measurements...")

# Extract important joints
pelvis = final_joints[PELVIS]
left_hip = final_joints[LEFT_HIP]
right_hip = final_joints[RIGHT_HIP]
left_knee = final_joints[LEFT_KNEE]
right_knee = final_joints[RIGHT_KNEE]
neck = final_joints[NECK]
left_shoulder = final_joints[LEFT_SHOULDER]
right_shoulder = final_joints[RIGHT_SHOULDER]

# Compute chest center as midpoint between shoulders
chest_point = (left_shoulder + right_shoulder) / 2.0

# Compute waist center as midpoint between chest and pelvis
waist_point = (chest_point + pelvis) / 2.0

# Compute hip center as midpoint between left and right hip
hip_point = (left_hip + right_hip) / 2.0

# Compute left thigh center as midpoint between left hip and left knee
left_thigh_point = (left_hip + left_knee) / 2.0

# Compute body axis from pelvis to neck
body_axis = normalize_vector(neck - pelvis)

# Compute left thigh axis from hip to knee
left_thigh_axis = normalize_vector(left_knee - left_hip)

# Print landmarks
print(f"Chest point: {chest_point}")
print(f"Waist point: {waist_point}")
print(f"Hip point: {hip_point}")
print(f"Left thigh point: {left_thigh_point}")
print(f"Body axis: {body_axis}")
print(f"Left thigh axis: {left_thigh_axis}")

# Compute circumferences in model units
chest_circumference = section_circumference(fitted_smpl_mesh, chest_point, body_axis)
waist_circumference = section_circumference(fitted_smpl_mesh, waist_point, body_axis)
hip_circumference = section_circumference(fitted_smpl_mesh, hip_point, body_axis)
thigh_circumference = section_circumference(fitted_smpl_mesh, left_thigh_point, left_thigh_axis)

# Convert circumferences to cm if available
chest_circumference_cm = to_cm(chest_circumference) if chest_circumference is not None else None
waist_circumference_cm = to_cm(waist_circumference) if waist_circumference is not None else None
hip_circumference_cm = to_cm(hip_circumference) if hip_circumference is not None else None
thigh_circumference_cm = to_cm(thigh_circumference) if thigh_circumference is not None else None

# Print final answers
print("\nFinal Circumference Measurements (cm)")
print("-------------------------------------")

if waist_circumference_cm is not None:
    print(f"Waist circumference (cm): {waist_circumference_cm:.2f}")
else:
    print("Waist circumference (cm): Could not compute")

if hip_circumference_cm is not None:
    print(f"Hip circumference (cm): {hip_circumference_cm:.2f}")
else:
    print("Hip circumference (cm): Could not compute")

if chest_circumference_cm is not None:
    print(f"Chest circumference (cm): {chest_circumference_cm:.2f}")
else:
    print("Chest circumference (cm): Could not compute")

if thigh_circumference_cm is not None:
    print(f"Thigh circumference (cm): {thigh_circumference_cm:.2f}")
else:
    print("Thigh circumference (cm): Could not compute")

Computing final circumference measurements...
Chest point: [ 0.01028093 -0.48425642  0.10937299]
Waist point: [-0.01190453 -0.67127395  0.06778056]
Hip point: [-0.01755708 -0.94979954  0.00316702]
Left thigh point: [ 0.15611538 -1.1182859  -0.04297348]
Body axis: [0.18897311 0.9463715  0.2620497 ]
Left thigh axis: [ 0.51628923 -0.8394053  -0.1698356 ]
Trying to compute section circumference...
Section found at offset 0.0000 with circumference 1.0671
Trying to compute section circumference...
Section found at offset 0.0000 with circumference 1.1399
Trying to compute section circumference...
Section found at offset 0.0000 with circumference 1.5428
Trying to compute section circumference...
Section found at offset 0.0000 with circumference 1.1996

Final Circumference Measurements (cm)
-------------------------------------
Waist circumference (cm): 113.99
Hip circumference (cm): 154.28
Chest circumference (cm): 106.71
Thigh circumference (cm): 119.96


/tmp/ipykernel_5325/625605652.py:44: DeprecationWarning: DEPRECATED: replace `path.to_planar`->`path.to_2D), removal 1/1/2026
  planar_section, _ = section.to_planar()
